Design, Sampling, and Collection
================================

**Author:** Ethan Ligon



## Reading



-   Deaton ch. 1 — the whole chapter
-   Deaton §1.4 on survey design and §2.2 on what to do about it
-   Skim the GLSS7 *Report* methodology section

Deaton is open access (CC BY 3.0 IGO).  On the hub it is already in your
`reading/` folder; otherwise [PDF](https://documents.worldbank.org/curated/en/203811547671768139/pdf/133790-PUB.pdf).

The GLSS7 report is published by the Ghana Statistical Service and is not
ours to redistribute:
[GSS catalogue 97](https://microdata.statsghana.gov.gh/index.php/catalog/97).



## Setup



In [1]:
%xmode Plain

import lsms_library as ll
import pandas as pd

## Brief description of LSMS Surveys



A timeline of the surveys `lsms_library` knows about: one row per country,
one bar per survey wave, so we can see overall temporal coverage — 37
countries, 113 waves, 1985 to the present.



## Topical coverage of LSMS Surveys



You can check current coverage yourself:



In [1]:
cov = ll.coverage()
cov

`sane` means the library builds that feature for that wave and the result
passes its checks; `absent` and `undeclared` mean it is not there to build.

Summarizing:



In [1]:
print("countries: %d | features: %d | waves: %d"
      % (cov.country.nunique(), cov.feature.nunique(),
         cov[cov.wave != ""].wave.nunique()))
cov.tier.value_counts()

So what is available for a country you care about:



In [1]:
cov.head(10)

## LSMS\_Library is a work in progress



## Explore Example Country



What countries are available?



In [1]:
ll.countries()

I'll use Uganda as my example; it's one of the best developed countries in
`lsms_library`.  But I encourage you to play with others; just change the
string 'Uganda' below.



In [1]:
eg = ll.Country('Uganda')
eg.waves

What different kinds of features/data are available for this country?



In [1]:
eg.data_scheme

What does the sample look like?



In [1]:
eg.sample()

Some actual data:



In [1]:
eg.household_roster().sort_index()

## Survey Design



## Survey Design Example



## Survey Design Example



Consider the Ghana Living Standards Survey (GhanaLSS), and specifically the
most recent publicly released wave 7 (2016–17).

-   **Population:** 



In [1]:
glss = ll.Country('GhanaLSS')   
WAVE = '2016-17'

glss.population[WAVE].population_statement

-   **Sample:** 



In [1]:
glss7 = glss[WAVE]
glss7.sample()

-   **Sample Frame:** A list of private dwellings from the 2010 census.  What
    does this exclude?
-   **Sample Weights:** Proportional to the inverse of the *ex ante*
    probability that a given sample household was selected for inclusion in
    the sample.  The weights in `lsms_library` are normalized to have a mean
    of one, but sometimes instead they're normalized to sum to the estimated
    size of the population.
-   **Sample Data:** Actual information collected from actual households in the
    sample.  For example, here's some information on the individuals within
    each household in the GhanaLSS 2016-17 data:



In [1]:
glss7.household_roster()

There are many different similar tables available; to see them



In [1]:
glss.features

## Choosing a Population



## Choosing a Survey Frame



## Census approach to enumerating households



## Two-Stage Cluster Based Sampling



## Example: Two-Stage Sampling Weights



Guinea-Bissau provides a fairly simple example.  It stratifies by region (see
below); within a region it uses a simple two-stage sample.  Consider the
region Bafata; here is a map of the sample EAs chosen in the first stage.



In [1]:
ll.coordinate_map('Guinea-Bissau', where={'Region': 'bafata'})

## Sampling Weights



## Sampling Weights



## Sub-Populations



## Stratification



## Strata Weights



## Two-Stage Sampling Weights



## Example: Two-Stage Sampling Weights



Back to our Bafata example.



In [1]:
ll.coordinate_map('Guinea-Bissau', size='weight', where={'Region': 'bafata'})

In [1]:
s = ll.Country("Guinea-Bissau").sample().reset_index()
b = s[s["strata"] == "Bafata"]

cl = b.groupby("v").agg(w=("weight", "first"),
                        urban=("Rural", "first"),
                        take=("weight", "size"))
cl["w"] = cl["w"].astype(float)

print("clusters  :", len(cl))
print("households:", len(b))
print("take      :", dict(cl["take"].value_counts()))

Fifty clusters, 598 households, and forty-eight of the fifty contain exactly
twelve households — the two that hold eleven are non-response, not design.
This is the textbook object: draw clusters, take a fixed number from each.



In [1]:
cv = b.groupby("v")["weight"].std().fillna(0) / b.groupby("v")["weight"].mean()
print("clusters whose weight varies inside them:", int((cv > 1e-9).sum()), "of", len(cl))
print("weights: min %.3f  median %.3f  max %.3f" % (cl.w.min(), cl.w.median(), cl.w.max()))

The weight is constant within every cluster, but not *across* clusters.

-   Everyone in cluster selected with equal probability
-   But weights differ *across* clusters, which tells us different cluster are
    selected with *different* probabilities.



## Inferring sample design from weights



## Inferring sample design from weights



In [1]:
print("sum of weights over the whole survey:", round(s["weight"].sum(), 1))
print("households in the survey            :", len(s))

Equal — the library normalizes weights to mean 1, so they carry *relative*
probabilities and nothing about population totals.  Everything below is a
ratio.

Perhaps the fifty clusters were drawn by **simple random sampling** from
Bafata's enumeration areas, each equally likely.  Then $p_c = N/n$ is the
same for all of them and the identity collapses:

$$ w_i \;\propto\; \frac{M_i}{b_i} $$

A household in a large EA is less likely to be drawn, so it stands for more
people.



In [1]:
cl["M_rel"] = cl["w"] * cl["take"]
cl["M_rel"] = cl["M_rel"] / cl["M_rel"].median()

print(cl["M_rel"].describe(percentiles=[.05, .25, .5, .75, .95]).round(2).to_string())

Sanity check :  Half the clusters fall between 0.89 and 1.04 times the median. Enumeration areas are usually *built* to be roughly equal in size.



## Recovering both probabilities



In [1]:
cl["pi_rel"] = 1 / cl["w"]                    # overall, relative
cl["stage2"] = cl["take"] / cl["M_rel"]       # within-cluster, relative
cl["stage1"] = cl["pi_rel"] / cl["stage2"]    # what is left over

print("stage-1 probability across the 50 clusters:")
print("  cv = %.2e  (SRS says it should be constant)" % (cl["stage1"].std() / cl["stage1"].mean()))

Zero to numerical precision.  This is about the simplest case of two-stage
cluster sampling:

1.  EAs drawn with equal probability ($r_c =1/\text{Number of EAs}$)
2.  Households within a EA drawn with equal probability ($p_i=1/M_{c_i}$)



## Hazards of ignoring sample weights



Consider the following **Population Pyramid**, a nice graphical device for
thinking about the demographic structure of a population.  Because we're
after population statistics, here we use the sample weights:



In [1]:
from lsms_library.visualizations import population_pyramid

ax = population_pyramid(glss, wave=WAVE, weights=True)
ax.figure.set_size_inches(7, 5.5)

## Hazards of ignoring sample weights



Now, contrast if we *neglect* the survey weights, and just treat our data as
if it was a simple random sample:



In [1]:
ax = population_pyramid(glss, wave=WAVE, weights=True, ghost=False)
ax.figure.set_size_inches(7, 5.5)

The filled bars are weighted; the outline is unweighted.  The outline stands
proud of the fill at every young age band and hugs it at the old ones, so the
unweighted sample contains proportionally **more children** than the population
it represents.



## Why?



Let's consider two facts.



In [1]:
r = glss.household_roster(waves=[WAVE]).reset_index()
s = glss.sample(waves=[WAVE]).reset_index()

people = len(r)
weighted_total = s.set_index("i")["weight"].reindex(r["i"]).sum()
print(f"people in the roster      : {people:,}")
print(f"sum of their weights      : {weighted_total:,.1f}")
print(f"mean weight per household : {s['weight'].mean():.4f}")

The mean weight is 1 by construction.  But that mean is over **households**
while the roster is one row per **person**.  So the weighted person-total is
$ \sum_h n_h w_h $, which equals the headcount only if weight and household
size are uncorrelated.

So are they uncorrelated?



In [1]:
size = r.groupby("i").size().rename("hhsize")
d = s.set_index("i").join(size).dropna(subset=["hhsize"])
print("corr(weight, household size) =", round(d["weight"].corr(d["hhsize"]), 3))
print()
print(d.groupby(pd.cut(d.hhsize, [0, 1, 2, 3, 5, 8, 50],
                       labels=["1", "2", "3", "4-5", "6-8", "9+"]),
               observed=True)
       .agg(households=("weight", "size"), mean_weight=("weight", "mean"))
       .round(3).to_string())

Bigger households carry smaller weights.  Since bigger households hold more
children, an unweighted count over-represents the young.

